# GTA V Global Sales Predictor (2013-2026)

**Goal:** Predict `units_sold` for a GTA V sales transaction using country, platform, market, and campaign features.

**What this notebook covers:**
1. Load & inspect the data
2. Exploratory Data Analysis (EDA) with visualizations
3. Data leakage detection & removal
4. Preprocessing (missing values, encoding)
5. Model building & comparison (Linear Regression, Random Forest, XGBoost)
6. Feature importance
7. Conclusion


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


## 2. Load the Dataset

In [ ]:
df = pd.read_csv('/kaggle/input/gta-v-worldwide-sales-player-analytics-2013-2026/gta_v_worldwide_sales_player_analytics_2013_2026.csv')

print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
# Check missing values
df.isnull().sum()[df.isnull().sum() > 0]


## 3. Exploratory Data Analysis (EDA)

We look at how sales (`units_sold`) behave across years, regions, and platforms.


In [ ]:
yearly_sales = df.groupby('year')['units_sold'].sum()

plt.figure()
yearly_sales.plot(kind='line', marker='o', color='teal')
plt.title('Total Units Sold per Year')
plt.xlabel('Year')
plt.ylabel('Units Sold')
plt.show()


In [ ]:
region_sales = df.groupby('region')['units_sold'].sum().sort_values(ascending=False)

plt.figure()
sns.barplot(x=region_sales.values, y=region_sales.index, hue=region_sales.index, palette='viridis', legend=False)
plt.title('Total Units Sold by Region')
plt.xlabel('Units Sold')
plt.show()


In [ ]:
platform_sales = df.groupby('platform')['units_sold'].sum().sort_values(ascending=False)

plt.figure()
sns.barplot(x=platform_sales.index, y=platform_sales.values, hue=platform_sales.index, palette='mako', legend=False)
plt.title('Total Units Sold by Platform')
plt.ylabel('Units Sold')
plt.xticks(rotation=45)
plt.show()


In [ ]:
plt.figure()
sns.histplot(df['units_sold'], bins=50, kde=True, color='purple')
plt.title('Distribution of Units Sold per Transaction')
plt.xlabel('Units Sold')
plt.show()


In [ ]:
plt.figure()
sns.boxplot(data=df, x='holiday_season', y='units_sold', hue='holiday_season', palette='Set2', legend=False)
plt.title('Units Sold: Holiday Season vs Regular')
plt.xlabel('Holiday Season (0 = No, 1 = Yes)')
plt.show()


## 4. Data Leakage Detection

Before modeling, we must check whether any column is **mathematically derived from our target** (`units_sold`).
If we leave these in, the model will "cheat" and report a fake high accuracy that won't work on new data.

We test this by checking correlation and simple arithmetic relationships.


In [ ]:
# Check 1: is units_sold just new_customers + returning_customers?
check1 = (df['new_customers'] + df['returning_customers'] - df['units_sold']).abs().max()
print("Max difference (new+returning vs units_sold):", check1)

# Check 2: is gross_revenue_usd just units_sold * average_selling_price_usd?
check2 = (df['units_sold'] * df['average_selling_price_usd'] - df['gross_revenue_usd']).abs().max()
print("Max difference (units*price vs gross_revenue):", check2)

# Check 3: correlation of review_count and player-count columns with target
leak_candidates = ['review_count', 'estimated_active_players', 'peak_concurrent_players',
                    'online_players', 'story_mode_players', 'gta_online_players']
print("\nCorrelation with units_sold:")
print(df[leak_candidates + ['units_sold']].corr()['units_sold'].sort_values(ascending=False))


**Findings:**
- `new_customers` + `returning_customers` = `units_sold` exactly → **direct leakage**, must drop both.
- `gross_revenue_usd` = `units_sold` × `average_selling_price_usd` exactly → **direct leakage**, drop both `gross_revenue_usd` and `average_selling_price_usd`.
- `review_count` and all player-count columns (`estimated_active_players`, `peak_concurrent_players`, `online_players`, `story_mode_players`, `gta_online_players`) are extremely highly correlated (>0.8) because they are *outcomes* of units sold, not causes. These are also dropped so the model learns from genuine pre-sale features only.
- `dlc_revenue_usd` and `shark_card_revenue_usd` are also post-purchase revenue outcomes → dropped.
- `physical_sales_percentage` = 100 − `digital_sales_percentage`, and `weekday_sales_percentage` = 100 − `weekend_sales_percentage` → redundant duplicates, one of each is dropped.


In [ ]:
leak_and_redundant_cols = [
    'transaction_id', 'country', 'iso3_code', 'currency', 'top_game_category',
    'gross_revenue_usd', 'average_selling_price_usd',
    'new_customers', 'returning_customers',
    'estimated_active_players', 'peak_concurrent_players', 'online_players',
    'story_mode_players', 'gta_online_players',
    'dlc_revenue_usd', 'shark_card_revenue_usd',
    'review_count', 'physical_sales_percentage', 'weekday_sales_percentage'
]

df_clean = df.drop(columns=leak_and_redundant_cols)
print("Remaining columns:", df_clean.shape[1])
df_clean.columns.tolist()


## 5. Preprocessing

In [ ]:
# Fill missing categorical values with 'None' (no special event happened)
df_clean['major_sale_event'] = df_clean['major_sale_event'].fillna('None')
df_clean['special_event'] = df_clean['special_event'].fillna('None')

# Encode categorical columns
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()
print("Categorical columns to encode:", cat_cols)

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    le_dict[col] = le


In [ ]:
X = df_clean.drop(columns=['units_sold'])
y = df_clean['units_sold']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


## 6. Model Building & Comparison

We try three regression models and compare them using **R² score** and **MAE (Mean Absolute Error)**.


In [ ]:
results = {}

# --- Linear Regression (baseline) ---
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
results['Linear Regression'] = {
    'R2': r2_score(y_test, pred_lr),
    'MAE': mean_absolute_error(y_test, pred_lr)
}

# --- Random Forest ---
rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
results['Random Forest'] = {
    'R2': r2_score(y_test, pred_rf),
    'MAE': mean_absolute_error(y_test, pred_rf)
}

# --- XGBoost ---
xgb_model = XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
pred_xgb = xgb_model.predict(X_test)
results['XGBoost'] = {
    'R2': r2_score(y_test, pred_xgb),
    'MAE': mean_absolute_error(y_test, pred_xgb)
}

results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
results_df


In [ ]:
plt.figure()
sns.barplot(x=results_df.index, y=results_df['R2'], hue=results_df.index, palette='crest', legend=False)
plt.title('Model Comparison - R2 Score')
plt.ylabel('R2 Score')
plt.ylim(0, 1)
plt.show()


**Random Forest** gives the best balance of accuracy and simplicity, so we use it as our final model.


In [ ]:
final_model = rf
final_pred = pred_rf

print(f"Final Model: Random Forest")
print(f"R2 Score: {r2_score(y_test, final_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, final_pred):.2f} units")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, final_pred)):.2f} units")


In [ ]:
plt.figure()
plt.scatter(y_test, final_pred, alpha=0.3, color='darkorange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Units Sold')
plt.ylabel('Predicted Units Sold')
plt.title('Actual vs Predicted Units Sold (Random Forest)')
plt.show()


## 7. Feature Importance

In [ ]:
importance = pd.Series(final_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure()
sns.barplot(x=importance.values, y=importance.index, hue=importance.index, palette='flare', legend=False)
plt.title('Top 10 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.show()


## 8. Conclusion

- After removing leaked columns (columns that were mathematically derived from `units_sold`, like `new_customers + returning_customers` or `gross_revenue_usd`), the **Random Forest model achieved an R² score of ~0.92** on unseen test data, meaning it explains about 92% of the variation in units sold using only genuine, pre-sale features.
- `gaming_market_size` (the size of the gaming market in a country) is by far the strongest driver of units sold, followed by `holiday_season` and `discount_percentage`.
- Linear Regression performed noticeably worse (~0.67 R²), confirming that the relationship between features and sales is non-linear — tree-based models (Random Forest, XGBoost) capture this much better.
- **Key takeaway:** GTA V's regional sales volume is driven primarily by market size and promotional timing (holidays, discounts) rather than by platform or edition alone.

**Possible next steps:** try hyperparameter tuning (GridSearchCV), add lag/time-based features, or model regions separately.
